# 02 · The OpenAI Agents SDK

**Module 06 · Agent Frameworks**

The **OpenAI Agents SDK** is the lightest of the three frameworks. It has only a few ideas:

| Idea | What it is |
|---|---|
| **Agent** | A name, instructions, a model and tools |
| **Tool** | A plain Python function (wrapped with `function_tool`) |
| **Runner** | Runs the agent loop for you |
| **Handoff** | One agent passes the conversation to another |

If you finished Modules 02 and 04, this will feel familiar. It is the same ideas with the plumbing removed.

In [1]:
from agents import Agent, Runner, function_tool, set_tracing_disabled
import shop_tools as shop

# The SDK can send traces of every run to your OpenAI dashboard. We switch that off to keep things simple.
set_tracing_disabled(True)

print("Ready")

Ready


## 1. Your first agent
An agent is just a name and instructions. `Runner.run` runs the whole loop and gives us the answer.

> **Why `await`?** The SDK is built on Python's `asyncio`, and Jupyter already runs an event loop, so here we write `await Runner.run(...)`. In a normal script you can write `Runner.run_sync(...)` instead (the desk files do that).

In [2]:
greeter = Agent(
    name="Greeter",
    instructions="You work at an online shop. Reply in one friendly sentence.",
    model=shop.MODEL,
)

result = await Runner.run(greeter, "Hi! I just placed my first order. Anything I should know?")
print(result.final_output)

Hi there! Thank you for your order; you'll receive a confirmation email shortly, and if you have any questions, feel free to reach out!


## 2. Give it a tool
In Module 02 we wrote a block of JSON for each tool. Now we pass our plain function to `function_tool`, and the SDK reads the name, the docstring and the type hints to build the same description.

Here is what it built for `lookup_order`:

In [3]:
lookup_tool = function_tool(shop.lookup_order)

print("Name:", lookup_tool.name)
print("Description:", lookup_tool.description)
print("Inputs:", lookup_tool.params_json_schema["properties"])

Name: lookup_order
Description: Get the details of one order: item, price, status and delivery date.
Inputs: {'order_id': {'title': 'Order Id', 'type': 'string'}}


That is the same information as our hand-written JSON, built from the function. **Write a good docstring: it is the model's manual for the tool.**

Now an agent that uses it:

In [4]:
orders_agent = Agent(
    name="Orders agent",
    instructions=shop.ORDERS_INSTRUCTIONS,
    tools=[lookup_tool],
    model=shop.MODEL,
)

shop.reset_shop()
result = await Runner.run(orders_agent, "Hi, can you tell me where my order A1002 is?")
print(result.final_output)

Your order for running shoes is currently in transit and is expected to be delivered by November 25, 2026.


## 3. Look inside a run
The result keeps a record of everything that happened. This is how you debug an agent.

In [5]:
for item in result.new_items:
    print(type(item).__name__)

print()
print("Tools that ran:", shop.TOOL_CALLS)
print("Tokens used:", result.context_wrapper.usage.total_tokens)

ToolCallItem
ToolCallOutputItem
MessageOutputItem

Tools that ran: ['lookup_order']
Tokens used: 317


You can see the tool call, the tool's output and the final message. In Module 02 we printed these ourselves. Here the framework records them.

## 4. Handoffs: a triage agent and three specialists
A **handoff** lets one agent pass the customer to another. We make a **triage agent** whose only job is to choose. The code below is the desk from `agents_sdk_desk.py`.

Each specialist has a `handoff_description`. That is what the triage agent reads to decide who gets the email.

In [6]:
# A tool is just our plain function wrapped with function_tool. The SDK reads the name, the docstring
# and the type hints and builds the tool description for us.
orders_agent = Agent(
    name="Orders agent",
    instructions=shop.ORDERS_INSTRUCTIONS,
    tools=[function_tool(shop.lookup_order)],
    model=shop.MODEL,
    handoff_description="Questions about where an order is or what is in it",  # what the triage agent reads
)

policy_agent = Agent(
    name="Policy agent",
    instructions=shop.POLICY_INSTRUCTIONS,
    tools=[function_tool(shop.search_policies)],
    model=shop.MODEL,
    handoff_description="Questions about shop rules: shipping, warranty, payments",
)

refunds_agent = Agent(
    name="Refunds agent",
    instructions=shop.REFUNDS_INSTRUCTIONS,
    tools=[function_tool(shop.check_refund_eligibility), function_tool(shop.issue_refund)],
    model=shop.MODEL,
    handoff_description="Customers who want their money back",
)

# The triage agent does not answer. It hands the customer over to the right specialist.
triage_agent = Agent(
    name="Triage agent",
    instructions="Send the customer's email to the right specialist. Do not answer it yourself.",
    handoffs=[orders_agent, policy_agent, refunds_agent],
    model=shop.MODEL,
)

In [7]:
emails = [
    "Hi, can you tell me where my order A1002 is?",
    "How much does express shipping cost?",
    "The left earbud on my headphones (order A1001) stopped working. I would like my money back.",
]

for email in emails:
    shop.reset_shop()
    result = await Runner.run(triage_agent, email)
    print("Email:", email)
    print("  Handled by:", result.last_agent.name)
    print("  Tools that ran:", shop.TOOL_CALLS)
    print("  Refunds issued:", shop.REFUNDS)
    print("  Reply:", result.final_output)
    print("  Tokens:", result.context_wrapper.usage.total_tokens)
    print()

Email: Hi, can you tell me where my order A1002 is?
  Handled by: Orders agent
  Tools that ran: ['lookup_order']
  Refunds issued: []
  Reply: Your order of running shoes is currently in transit and is expected to be delivered by November 25, 2026. The total price is $120.
  Tokens: 545



Email: How much does express shipping cost?
  Handled by: Policy agent
  Tools that ran: ['search_policies']
  Refunds issued: []
  Reply: Express shipping always costs $12. Standard shipping is free on orders over $50.
  Tokens: 680



Email: The left earbud on my headphones (order A1001) stopped working. I would like my money back.
  Handled by: Refunds agent
  Tools that ran: ['check_refund_eligibility', 'issue_refund']
  Refunds issued: [{'order_id': 'A1001', 'amount': 79.0}]
  Reply: Your refund of $79.00 for order A1001 has been issued. Please allow a few days for it to reflect in your account.
  Tokens: 1048



The triage agent did not answer. It handed over, and `result.last_agent` tells us who gave the final answer.

Compare with Module 04: there we wrote the router as a separate function with a structured output. Here the router is **built in** as the `handoffs` list. Fewer lines, but the routing decision happens **inside the model's tool calls**, so you see less of it.

## 5. What else the SDK includes
We keep this module simple, but you should know these exist. They are the reason people pick the SDK.

| Feature | What it does | Connects to |
|---|---|---|
| **Guardrails** | Input and output checks attached to an agent | Module 05 |
| **Tracing** | Every run recorded and viewable in your OpenAI dashboard (we switched it off above) | Debugging |
| **Sessions** | Built-in conversation memory | Module 03 |
| **Streaming** | Show the answer as it is written | Module 01 |
| **MCP tools** | Plug in tools from other servers | Module 02 |

## 6. What it costs
- **Still below version 1.0.** The SDK changes between releases (for example, the default model has changed). That is why `requirements.txt` pins `openai-agents`.
- **A handoff costs a model call.** The triage agent is an extra call, which is why this desk uses more tokens than the hand-built one (notebook 05 measures it).
- **Less visible routing.** The model decides who gets the email, and you cannot easily say *why*.
- **Works best with OpenAI models.** Other providers are supported, with extra setup.

**Pick it when** you want the quickest path from nothing to a working agent system on OpenAI models.

Next: **LangGraph**, where *you* draw the flow.